# 🎤 RVC (Retrieval-based Voice Conversion) CLI Colab 학습 가이드

이 노트북은 **Gradio WebUI 없이** 명령어(CLI / Python Script)만으로 Google Colab 환경에서 RVC v2 모델을 학습시키는 가이드입니다.

### 📌 전체 진행 순서
1. **GPU & 환경 설정**
2. **구글 드라이브 연동 & 필수 사전 학습(Pretrained) 모델 다운로드**
3. **데이터셋 업로드 및 압축 해제**
4. **데이터 전처리 (Audio Splitting & Resampling)**
5. **특징(Pitch F0 & HuBERT Feature) 추출**
6. **Filelist 및 Config.json 생성**
7. **PyTorch 모델 학습 (Train)**
8. **FAISS Feature Index 생성**
9. **완성된 Weights (.pth) & Index (.index) 구글 드라이브 내보내기**

---
## 1. GPU 및 환경 설정
Colab GPU 연결 상태를 확인하고, RVC 저장소 클론 및 필요한 라이브러리를 설치합니다.

In [ ]:
# 1. GPU 상태 확인
!nvidia-smi

import os
# 2. RVC 레포지토리 클론
if not os.path.exists("/content/Retrieval-based-Voice-Conversion-WebUI"):
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git /content/Retrieval-based-Voice-Conversion-WebUI

%cd /content/Retrieval-based-Voice-Conversion-WebUI

# 3. aria2 고속 다운로더 및 필수 패키지 설치
!apt-get update -qq && !apt-get install -y -qq aria2
!pip install -r requirements.txt
!pip install faiss-cpu fairseq praat-parselmouth pyworld torch-directml tensorboard

---
## 2. 구글 드라이브 연동 & 필수 사전 학습 모델 다운로드
학습 결과물 저장 및 데이터셋 로드를 위해 Google Drive를 마운트하고, HuBERT, RMVPE 및 Pretrained Baseline 모델을 다운로드합니다.

In [ ]:
from google.colab import drive
import os

# 1. 구글 드라이브 마운트
drive.mount("/content/drive")

# 2. 디렉토리 구조 생성
!mkdir -p assets/hubert assets/rmvpe assets/pretrained_v2 assets/indices assets/weights

# 3. HuBERT base 다운로드
if not os.path.exists("assets/hubert/hubert_base.pt"):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt -d assets/hubert -o hubert_base.pt

# 4. RMVPE Pitch Extractor 다운로드
if not os.path.exists("assets/rmvpe/rmvpe.pt"):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt -d assets/rmvpe -o rmvpe.pt

# 5. v2 사전 학습 모델 (40k 기준) 다운로드
if not os.path.exists("assets/pretrained_v2/f0G40k.pth"):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth -d assets/pretrained_v2 -o f0G40k.pth

if not os.path.exists("assets/pretrained_v2/f0D40k.pth"):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth -d assets/pretrained_v2 -o f0D40k.pth

print("✅ 필수 사전 학습 모델 다운로드 완료!")

---
## 3. 학습 데이터셋 준비
Google Drive 상에 올린 음성 데이터셋 압축 파일 (`dataset.zip`)을 풀어 `dataset` 폴더에 위치시킵니다.

In [ ]:
import os

# 💡 구글 드라이브 내 데이터셋 zip 파일 경로
ZIP_PATH = "/content/drive/MyDrive/dataset.zip"
DATASET_DIR = "/content/Retrieval-based-Voice-Conversion-WebUI/dataset"

!mkdir -p {DATASET_DIR}

if os.path.exists(ZIP_PATH):
    !unzip -q -o "{ZIP_PATH}" -d "{DATASET_DIR}"
    print(f"✅ {ZIP_PATH} 압축 해제 완료!")
else:
    print(f"⚠️ {ZIP_PATH} 경로를 찾을 수 없습니다. 경로를 수정하거나 dataset 디렉토리에 음성(.wav, .mp3) 파일을 직접 업로드해 주세요.")

---
## 4. 데이터 전처리 (Audio Splitting & Resampling)
입력 음성을 지정한 샘플레이트(예: 40000Hz)로 변환하고 적절한 시간 길이로 분할합니다.

In [ ]:
# ⚙️ 전처리 설정 변수
EXP_NAME = "my_rvc_model"  # 모델 및 실험 이름
SAMPLE_RATE = 40000        # 샘플레이트: 40000 (40k), 48000 (48k), 32000 (32k)
NUM_PROCESSES = 2          # CPU 프로세스 수 (Colab 환경 2 권장)
PREPROCESS_PER = 3.0       # 자르는 단위 (초)

LOG_DIR = f"logs/{EXP_NAME}"

# 데이터 전처리 스크립트 실행
!python train/preprocess.py dataset {SAMPLE_RATE} {NUM_PROCESSES} {LOG_DIR} False {PREPROCESS_PER}

print("✅ 데이터 전처리 완료!")

---
## 5. Pitch (F0) 및 HuBERT Feature 추출
GPU를 사용하여 RMVPE 기반 Pitch(F0)와 HuBERT Feature(v2 768차원)를 추출합니다.

In [ ]:
VERSION = "v2"  # 모델 버전: v1 또는 v2

# 1. Pitch (F0) 추출 (GPU RMVPE 방식)
!python train/dataset/extract_f0.py cuda 1 0 0 logs/{EXP_NAME} True

# 2. HuBERT Feature 추출 (GPU 16bit half precision)
!python train/dataset/extract_hubert_feature.py cuda 1 0 0 logs/{EXP_NAME} {VERSION} True

print("✅ F0 및 HuBERT Feature 추출 완료!")

---
## 6. Filelist.txt 및 Config.json 생성
추출된 특징 데이터를 매핑하는 `filelist.txt`를 생성하고 모델 구조에 맞는 `config.json`을 준비합니다.

In [ ]:
import os
import json
import random

SR_STR = "40k"      # 40k, 48k, 32k 중 선택
SPK_ID = 0
IF_F0 = True

now_dir = os.getcwd()
exp_dir = os.path.join(now_dir, "logs", EXP_NAME)
gt_wavs_dir = os.path.join(exp_dir, "0_gt_wavs")
feature_dir = os.path.join(exp_dir, "3_feature768" if VERSION == "v2" else "3_feature256")

# 공통 파일명 탐색
gt_names = set([n.rsplit('.', 1)[0] for n in os.listdir(gt_wavs_dir) if n.endswith('.wav')])
feat_names = set([n.rsplit('.', 1)[0] for n in os.listdir(feature_dir) if n.endswith('.npy')])
names = gt_names & feat_names

if IF_F0:
    f0_dir = os.path.join(exp_dir, "2a_f0")
    f0nsf_dir = os.path.join(exp_dir, "2b-f0nsf")
    f0_names = set([n.rsplit('.', 1)[0] for n in os.listdir(f0_dir) if n.endswith('.npy')])
    f0nsf_names = set([n.rsplit('.', 1)[0] for n in os.listdir(f0nsf_dir) if n.endswith('.npy')])
    names = names & f0_names & f0nsf_names

opt = []
for name in sorted(list(names)):
    if IF_F0:
        opt.append(f"{gt_wavs_dir}/{name}.wav|{feature_dir}/{name}.npy|{f0_dir}/{name}.wav.npy|{f0nsf_dir}/{name}.wav.npy|{SPK_ID}")
    else:
        opt.append(f"{gt_wavs_dir}/{name}.wav|{feature_dir}/{name}.npy|{SPK_ID}")

# Mute 샘플 무음 패딩 추가
fea_dim = 768 if VERSION == "v2" else 256
for _ in range(2):
    if IF_F0:
        opt.append(f"{now_dir}/logs/mute/0_gt_wavs/mute{SR_STR}.wav|{now_dir}/logs/mute/3_feature{fea_dim}/mute.npy|{now_dir}/logs/mute/2a_f0/mute.wav.npy|{now_dir}/logs/mute/2b-f0nsf/mute.wav.npy|{SPK_ID}")
    else:
        opt.append(f"{now_dir}/logs/mute/0_gt_wavs/mute{SR_STR}.wav|{now_dir}/logs/mute/3_feature{fea_dim}/mute.npy|{SPK_ID}")

random.shuffle(opt)

# filelist.txt 쓰기
filelist_path = os.path.join(exp_dir, "filelist.txt")
with open(filelist_path, "w", encoding="utf8") as f:
    f.write("\n".join(opt))

# config.json 복사
config_src = f"configs/{VERSION}/{SR_STR}.json"
config_dst = os.path.join(exp_dir, "config.json")
if os.path.exists(config_src):
    with open(config_src, "r", encoding="utf8") as f:
        config_data = json.load(f)
    with open(config_dst, "w", encoding="utf8") as f:
        json.dump(config_data, f, ensure_ascii=False, indent=4)

print(f"✅ filelist.txt 및 config.json 생성 완료! (학습 샘플 개수: {len(names)}개)")

---
## 7. RVC PyTorch 모델 학습
준비된 데이터셋으로 Neural Network 모델 학습을 진행합니다.

In [ ]:
# ⚙️ 학습 하이퍼파라미터 설정
BATCH_SIZE = 8             # Colab Free T4 GPU 기준 8~16 권장
TOTAL_EPOCH = 100          # 총 학습 에포크 (기본 100~300 권장)
SAVE_EPOCH = 20            # 몇 에포크마다 체크포인트 저장할지

PRETRAINED_G = "assets/pretrained_v2/f0G40k.pth"
PRETRAINED_D = "assets/pretrained_v2/f0D40k.pth"

# 모델 학습 파이프라인 실행
!python train/train.py \
  -e {EXP_NAME} \
  -sr {SR_STR} \
  -f0 1 \
  -bs {BATCH_SIZE} \
  -g 0 \
  -te {TOTAL_EPOCH} \
  -se {SAVE_EPOCH} \
  -pg {PRETRAINED_G} \
  -pd {PRETRAINED_D} \
  -l 1 \
  -c 1 \
  -sw 1 \
  -v {VERSION}

print("🎉 RVC 모델 학습 완료!")

---
## 8. FAISS Index (검색 인덱스) 생성
음조 변환 시 자연스러운 음색 복원을 위해 FAISS feature `.index` 파일을 생성합니다.

In [ ]:
OUTSIDE_INDEX_ROOT = "assets/indices"

# 인덱스 생성 실행
!python train/train_index.py {EXP_NAME} {VERSION} {OUTSIDE_INDEX_ROOT} {NUM_PROCESSES}

print("✅ FAISS Index 파일 생성 완료!")

---
## 9. 구글 드라이브로 결과물 내보내기
완성된 모델 파일(`.pth`)과 인덱스 파일(`.index`)을 내 Google Drive의 `RVC_Output` 폴더로 저장합니다.

In [ ]:
import shutil
import glob
import os

DRIVE_SAVE_PATH = "/content/drive/MyDrive/RVC_Output"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

# 1. .pth weights 파일 내보내기
pth_files = glob.glob(f"assets/weights/{EXP_NAME}*.pth") + glob.glob(f"weights/{EXP_NAME}*.pth")
copied_pth = 0
for pth in pth_files:
    dst = os.path.join(DRIVE_SAVE_PATH, os.path.basename(pth))
    shutil.copy(pth, dst)
    print(f"📦 [Weights 백업] {pth} -> {dst}")
    copied_pth += 1

# 2. .index 파일 내보내기
index_files = glob.glob(f"logs/{EXP_NAME}/added_*.index") + glob.glob(f"assets/indices/*{EXP_NAME}*.index")
copied_idx = 0
for idx in set(index_files):
    dst = os.path.join(DRIVE_SAVE_PATH, os.path.basename(idx))
    shutil.copy(idx, dst)
    print(f"🔍 [Index 백업] {idx} -> {dst}")
    copied_idx += 1

print(f"\n🎉 구글 드라이브 저장 완료! (Weights: {copied_pth}개, Index: {copied_idx}개)")
print(f"📁 드라이브 경로: {DRIVE_SAVE_PATH}")